# Gov Collect Entra

Entra is the compiler's target instruction set — security groups are the one
currency all four planes accept — so the correctness that matters most here is
**effective** membership. Nested groups mean the person who can create a Fabric
data agent is often several hops from the group the entitlement was written
against, and a collector that only records direct membership produces an
entitlement report that is confidently wrong.

In [ ]:
dry_run = True
lakehouse_name = "governance_lh"
# Only expand membership for groups this app manages, by default. Expanding the
# whole directory is a very different cost profile in a large tenant.
app_managed_only = True
max_pages = 50

In [ ]:
# --- inlined from collectors/shape_common.py (unit-tested offline) ---
from __future__ import annotations

import json
import uuid
from datetime import datetime, timezone
from typing import Any, Callable, Iterable, Sequence


def utcnow() -> datetime:
    return datetime.now(timezone.utc)


def new_run_id() -> str:
    return str(uuid.uuid4())


def as_str(value: Any) -> str | None:
    """Normalise an API scalar to a string, preserving a real absence as None.

    Collector tables are all-string on purpose: every plane has its own id
    format, and coercing them into typed columns is how a join silently starts
    returning nothing.
    """
    if value is None:
        return None
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, sort_keys=True)


def as_json(value: Any) -> str | None:
    """Stable JSON for a blob column. Sorted keys so diffs are meaningful."""
    if value is None:
        return None
    return json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)


def stamp(rows: Iterable[dict], run_id: str, scanned_at: datetime | None = None) -> list[dict]:
    """Attach the run provenance every `gov_actual_*` row carries."""
    when = scanned_at or utcnow()
    out = []
    for row in rows:
        enriched = dict(row)
        enriched["run_id"] = run_id
        enriched["scanned_at"] = when
        out.append(enriched)
    return out


class RunLedger:
    """Accumulates what a collector did, for the `gov_runs` row and the app.

    Errors are first-class: a collector that quietly drops an unreadable object
    produces a governance report that is wrong in the most dangerous direction —
    it under-reports access.
    """

    def __init__(self, collector: str, module: str, tier: str) -> None:
        self.run_id = new_run_id()
        self.collector = collector
        self.module = module
        self.tier = tier
        self.started_at = utcnow()
        self.finished_at: datetime | None = None
        self.errors: list[dict[str, str]] = []
        self.counts: dict[str, int] = {}

    def count(self, table: str, n: int) -> None:
        self.counts[table] = self.counts.get(table, 0) + n

    def error(self, scope: str, exc: BaseException | str) -> None:
        self.errors.append(
            {
                "scope": scope,
                "type": type(exc).__name__ if isinstance(exc, BaseException) else "Error",
                "message": str(exc),
            }
        )

    def finish(self) -> dict:
        self.finished_at = utcnow()
        return {
            "run_id": self.run_id,
            "collector": self.collector,
            "module": self.module,
            "tier": self.tier,
            "started_at": self.started_at,
            "finished_at": self.finished_at,
            "n_objects": sum(self.counts.values()),
            "n_errors": len(self.errors),
            "error_json": as_json(self.errors) if self.errors else None,
            "duration_s": (self.finished_at - self.started_at).total_seconds(),
        }

    def exit_value(self, *, dry_run: bool) -> dict:
        """Actuator-contract-shaped result (PLAN.md §14) for the app to parse."""
        return {
            "ok": True,
            "dry_run": dry_run,
            "run_id": self.run_id,
            "collector": self.collector,
            "module": self.module,
            "tier": self.tier,
            "counts": dict(self.counts),
            "n_errors": len(self.errors),
            "errors": self.errors[:20],
            "finished_at": (self.finished_at or utcnow()).isoformat(),
        }


def safe_each(
    items: Sequence[Any],
    fn: Callable[[Any], list[dict]],
    ledger: RunLedger,
    scope_of: Callable[[Any], str],
) -> list[dict]:
    """Map `fn` over `items`, recording per-item failures instead of raising."""
    rows: list[dict] = []
    for item in items:
        try:
            rows.extend(fn(item))
        except Exception as exc:  # noqa: BLE001 — a collector must never hard-fail
            ledger.error(scope_of(item), exc)
    return rows

In [ ]:
# --- inlined from collectors/runtime.py (unit-tested offline) ---
from __future__ import annotations

import json
import time
from typing import Any, Callable


class RestError(RuntimeError):
    def __init__(self, status: int, url: str, body: str) -> None:
        super().__init__(f"{status} {url}: {body[:400]}")
        self.status = status
        self.url = url


def fabric_client():
    """A `sempy` REST client for Fabric / Power BI, under the running identity."""
    import sempy.fabric as fabric  # type: ignore

    return fabric.FabricRestClient()


def rest_get(client, path: str, *, retries: int = 4) -> dict[str, Any]:
    """GET with backoff on 429/5xx.

    Admin APIs are rate-limited (25 req/min on some tenant-setting endpoints), and
    a nightly crawl that gives up on the first 429 silently under-reports — which
    is the worst possible failure mode for a governance inventory.
    """
    delay = 2.0
    last: Exception | None = None
    for _ in range(retries):
        response = client.get(path)
        if response.status_code == 200:
            return response.json() if response.text else {}
        if response.status_code in (429, 500, 502, 503, 504):
            retry_after = response.headers.get("Retry-After")
            time.sleep(float(retry_after) if retry_after else delay)
            delay = min(delay * 2, 60)
            last = RestError(response.status_code, path, response.text)
            continue
        raise RestError(response.status_code, path, response.text)
    raise last or RestError(0, path, "exhausted retries")


def graph_token(scope: str = "https://graph.microsoft.com/.default") -> str:
    """Delegated Graph token for the identity the notebook runs as."""
    import notebookutils  # type: ignore

    return notebookutils.credentials.getToken(scope)


def graph_get(token: str, url: str, *, retries: int = 4) -> dict[str, Any]:
    import urllib.error
    import urllib.request

    if not url.startswith("http"):
        url = f"https://graph.microsoft.com{url}"

    delay = 2.0
    for _ in range(retries):
        request = urllib.request.Request(url, headers={"Authorization": f"Bearer {token}"})
        try:
            with urllib.request.urlopen(request) as response:  # noqa: S310 - fixed host
                return json.loads(response.read().decode("utf-8"))
        except urllib.error.HTTPError as exc:
            if exc.code in (429, 500, 502, 503, 504):
                time.sleep(delay)
                delay = min(delay * 2, 60)
                continue
            raise RestError(exc.code, url, exc.read().decode("utf-8", "replace")) from exc
    raise RestError(0, url, "exhausted retries")


def graph_call(token: str, method: str, url: str, body: dict | None = None) -> dict[str, Any]:
    """Graph request with a method — the write-capable sibling of `graph_get`.

    Deliberately **not** retried on 5xx: a POST that may have partially applied
    must not be replayed blindly. The actuator's read-before-write makes a
    retry safe only after re-reading, and that is the caller's decision.
    """
    import urllib.error
    import urllib.request

    if not url.startswith("http"):
        url = f"https://graph.microsoft.com{url}"

    data = json.dumps(body).encode("utf-8") if body is not None else None
    request = urllib.request.Request(url, data=data, method=method.upper())
    request.add_header("Authorization", f"Bearer {token}")
    if data is not None:
        request.add_header("Content-Type", "application/json")

    try:
        with urllib.request.urlopen(request) as response:  # noqa: S310 - fixed host
            payload = response.read().decode("utf-8")
            return json.loads(payload) if payload else {}
    except urllib.error.HTTPError as exc:
        raise RestError(exc.code, url, exc.read().decode("utf-8", "replace")) from exc


def fabric_call(client, method: str, path: str, body: dict | None = None) -> dict[str, Any]:
    """Fabric REST with a method, through the `sempy` client.

    Same no-retry stance as `graph_call`, for the same reason.
    """
    verb = method.upper()
    if verb == "GET":
        response = client.get(path)
    elif verb == "POST":
        response = client.post(path, json=body or {})
    elif verb == "PATCH":
        response = client.patch(path, json=body or {})
    elif verb == "DELETE":
        response = client.delete(path)
    else:
        raise ValueError(f"unsupported method {method}")

    if response.status_code not in (200, 201, 202, 204):
        raise RestError(response.status_code, path, response.text)
    return response.json() if response.text else {}


def write_table(
    spark,
    lakehouse: str,
    table: str,
    rows: list[dict],
    *,
    dry_run: bool,
    log: Callable[[str, str, str], None],
) -> int:
    """Overwrite one `gov_actual_*` table with this run's rows.

    Overwrite, not append: these tables are a *snapshot of current reality*, and
    the run ledger plus `gov_audit` carry the history. An append-only actual-state
    table is how a drift engine starts comparing against last month.
    """
    if dry_run:
        log(table, "Planned", f"{len(rows)} rows")
        return len(rows)
    if not rows:
        log(table, "Skipped (no permission)", "no rows collected")
        return 0
    try:
        df = spark.createDataFrame(rows)
        df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
            f"{lakehouse}.{table}"
        )
        log(table, "Created", f"{len(rows)} rows")
        return len(rows)
    except Exception as exc:  # noqa: BLE001
        log(table, "Failed", f"{type(exc).__name__}: {exc}")
        return 0


def write_run_row(spark, lakehouse: str, summary: dict, *, dry_run: bool) -> None:
    if dry_run:
        return
    try:
        spark.createDataFrame([summary]).write.mode("append").option(
            "mergeSchema", "true"
        ).saveAsTable(f"{lakehouse}.gov_runs")
    except Exception as exc:  # noqa: BLE001
        print(f"gov_runs append failed: {exc}")


def finish(ledger, spark, lakehouse: str, *, dry_run: bool) -> str:
    summary = ledger.finish()
    write_run_row(spark, lakehouse, summary, dry_run=dry_run)
    result = ledger.exit_value(dry_run=dry_run)
    try:
        import notebookutils  # type: ignore

        notebookutils.notebook.exit(json.dumps(result))
    except ImportError:
        print(json.dumps(result, indent=2))
    return json.dumps(result)

In [ ]:
# --- inlined from collectors/shape_entra.py (unit-tested offline) ---
from __future__ import annotations

from typing import Any, Callable, Iterable



#: Groups the app created and therefore may manage. Anything else is read-only
#: to us — a governance tool that edits groups it did not create is a liability.
APP_MANAGED_PREFIX = "GOV-"


def shape_groups(payload: dict[str, Any]) -> list[dict]:
    rows: list[dict] = []
    for group in payload.get("value", []) or []:
        name = as_str(group.get("displayName")) or ""
        types = group.get("groupTypes") or []
        rows.append(
            {
                "group_id": as_str(group.get("id")),
                "display_name": name,
                "mail": as_str(group.get("mail")),
                "group_type": "Microsoft365" if "Unified" in types else "Security",
                "security_enabled": as_str(group.get("securityEnabled")),
                "is_app_managed": as_str(name.startswith(APP_MANAGED_PREFIX)),
                "description": as_str(group.get("description")),
            }
        )
    return rows


def shape_group_members(group_id: str, payload: dict[str, Any]) -> list[dict]:
    """Direct members of one group.

    `#microsoft.graph.group` members are nested groups — kept, because they are
    edges in the membership graph, not leaves.
    """
    rows: list[dict] = []
    for member in payload.get("value", []) or []:
        odata_type = str(member.get("@odata.type", ""))
        if odata_type.endswith("user"):
            principal_type = "User"
        elif odata_type.endswith("group"):
            principal_type = "Group"
        elif odata_type.endswith("servicePrincipal"):
            principal_type = "ServicePrincipal"
        else:
            principal_type = "Other"
        rows.append(
            {
                "group_id": as_str(group_id),
                "principal_id": as_str(member.get("id")),
                "principal_type": principal_type,
                "principal_name": as_str(
                    member.get("displayName") or member.get("userPrincipalName")
                ),
                "is_transitive": "false",
            }
        )
    return rows


def resolve_transitive_members(
    direct: dict[str, list[dict]],
    *,
    max_depth: int = 20,
) -> list[dict]:
    """Expand nested groups into effective membership.

    `direct` maps group_id → its direct member rows.

    Returns one row per (group, effective principal), with `is_transitive`
    marking members that are only reachable through a nested group and `depth`
    recording how far away they are.

    Cycles are survivable: Entra permits them and a naive walk would hang. The
    visited-set is per starting group, and `max_depth` is a second belt.
    """
    resolved: list[dict] = []

    for root, _ in direct.items():
        seen_groups: set[str] = {root}
        # (group_to_expand, depth_of_its_members)
        frontier: list[tuple[str, int]] = [(root, 0)]
        seen_principals: set[str] = set()

        while frontier:
            current, depth = frontier.pop(0)
            if depth > max_depth:
                break
            for member in direct.get(current, []):
                principal_id = member.get("principal_id")
                if not principal_id:
                    continue
                principal_type = member.get("principal_type")

                if principal_type == "Group":
                    if principal_id not in seen_groups:
                        seen_groups.add(principal_id)
                        frontier.append((principal_id, depth + 1))
                    # A nested group is itself a member — keep the edge.
                if principal_id in seen_principals:
                    continue
                seen_principals.add(principal_id)
                resolved.append(
                    {
                        "group_id": root,
                        "principal_id": principal_id,
                        "principal_type": principal_type,
                        "principal_name": member.get("principal_name"),
                        "is_transitive": as_str(depth > 0),
                        "depth": as_str(depth),
                    }
                )

    return resolved


def shape_licenses(
    users_payload: dict[str, Any],
    sku_names: dict[str, str] | None = None,
) -> list[dict]:
    """User → licence rows.

    `assigned_via` distinguishes a direct assignment from group-based licensing,
    because only the latter is something an entitlement can compile onto.
    """
    names = sku_names or {}
    rows: list[dict] = []
    for user in users_payload.get("value", []) or []:
        user_id = as_str(user.get("id"))
        states = {
            str(s.get("skuId")): s
            for s in (user.get("licenseAssignmentStates") or [])
            if s.get("skuId")
        }
        for licence in user.get("assignedLicenses") or []:
            sku_id = as_str(licence.get("skuId"))
            state = states.get(str(sku_id), {})
            assigned_by_group = state.get("assignedByGroup")
            rows.append(
                {
                    "principal_id": user_id,
                    "principal_name": as_str(
                        user.get("userPrincipalName") or user.get("displayName")
                    ),
                    "sku_id": sku_id,
                    "sku_name": as_str(names.get(str(sku_id))),
                    "assigned_via": "Group" if assigned_by_group else "Direct",
                    "group_id": as_str(assigned_by_group),
                    "disabled_plans_json": as_json(licence.get("disabledPlans") or []),
                }
            )
    return rows


def index_sku_names(payload: dict[str, Any]) -> dict[str, str]:
    """`GET /v1.0/subscribedSkus` → {skuId: skuPartNumber}."""
    return {
        str(sku.get("skuId")): str(sku.get("skuPartNumber"))
        for sku in payload.get("value", []) or []
        if sku.get("skuId")
    }


def paged(
    fetch: Callable[[str], dict[str, Any]],
    first_url: str,
    *,
    max_pages: int = 50,
) -> Iterable[dict[str, Any]]:
    """Follow `@odata.nextLink`, with a hard page cap.

    An unbounded follow against a large directory is how a nightly job becomes a
    six-hour job; the cap is reported by the caller as a partial run rather than
    silently truncating.
    """
    url = first_url
    for _ in range(max_pages):
        payload = fetch(url)
        yield payload
        url = payload.get("@odata.nextLink")
        if not url:
            return

In [ ]:
steps = []


def log(step, status, detail=""):
    steps.append({"step": step, "status": status, "detail": detail})
    print(f"[{status:>22}] {step}{(' — ' + detail) if detail else ''}")


ledger = RunLedger("Gov Collect Entra", "entra", "T1")
token = graph_token()
print(f"run_id={ledger.run_id} dry_run={dry_run} app_managed_only={app_managed_only}")

## Groups

In [ ]:
group_rows = []
try:
    fetch = lambda url: graph_get(token, url)  # noqa: E731
    url = "/v1.0/groups?$top=999&$select=id,displayName,mail,groupTypes,securityEnabled,description"
    pages = 0
    for payload in paged(fetch, url, max_pages=max_pages):
        group_rows.extend(shape_groups(payload))
        pages += 1
    if pages >= max_pages:
        ledger.error("groups", f"page cap {max_pages} reached — inventory is partial")
    ledger.count("gov_actual_entra_groups", len(group_rows))
    log("groups", "Created", f"{len(group_rows)} groups over {pages} pages")
except Exception as exc:  # noqa: BLE001
    ledger.tier = "T0"
    ledger.error("groups", exc)
    log("groups", "Skipped (no permission)", str(exc))

## Membership

Direct membership first, then transitive expansion offline. Graph does offer
`/transitiveMembers`, but resolving locally keeps the *depth* and the
direct-vs-inherited distinction, which is what makes a derivation path
explainable in the Can-Do Explorer later.

In [ ]:
targets = [
    g for g in group_rows if not app_managed_only or g.get("is_app_managed") == "true"
]
log("membership scope", "Planned", f"{len(targets)} of {len(group_rows)} groups")

direct: dict[str, list[dict]] = {}
for group in targets:
    group_id = group.get("group_id")
    if not group_id:
        continue
    try:
        payload = graph_get(
            token,
            f"/v1.0/groups/{group_id}/members?$top=999"
            "&$select=id,displayName,userPrincipalName",
        )
        direct[group_id] = shape_group_members(group_id, payload)
    except Exception as exc:  # noqa: BLE001
        ledger.error(f"members:{group.get('display_name') or group_id}", exc)

member_rows = resolve_transitive_members(direct)
ledger.count("gov_actual_entra_group_members", len(member_rows))
log(
    "membership",
    "Created",
    f"{len(member_rows)} effective rows from {len(direct)} groups",
)

## Licences

`assigned_via` is the governance-relevant column: only **group-based**
licensing is something an entitlement can compile onto.

In [ ]:
licence_rows = []
try:
    skus = index_sku_names(graph_get(token, "/v1.0/subscribedSkus"))
    fetch = lambda url: graph_get(token, url)  # noqa: E731
    url = (
        "/v1.0/users?$top=999"
        "&$select=id,userPrincipalName,displayName,assignedLicenses,licenseAssignmentStates"
    )
    pages = 0
    for payload in paged(fetch, url, max_pages=max_pages):
        licence_rows.extend(shape_licenses(payload, skus))
        pages += 1
    if pages >= max_pages:
        ledger.error("licenses", f"page cap {max_pages} reached — inventory is partial")
    ledger.count("gov_actual_licenses", len(licence_rows))
    log("licences", "Created", f"{len(licence_rows)} assignments")
except Exception as exc:  # noqa: BLE001
    ledger.error("licenses", exc)
    log("licences", "Skipped (no permission)", str(exc))

## Write

In [ ]:
TABLES = [
    ("gov_actual_entra_groups", group_rows),
    ("gov_actual_entra_group_members", member_rows),
    ("gov_actual_licenses", licence_rows),
]

for table, rows in TABLES:
    write_table(
        spark,  # noqa: F821
        lakehouse_name,
        table,
        stamp(rows, ledger.run_id),
        dry_run=dry_run,
        log=log,
    )

finish(ledger, spark, lakehouse_name, dry_run=dry_run)  # noqa: F821